In [8]:
import re
import os
import csv
import time
import math
import json
import requests  
import numpy as np
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup # # html解析库
from selenium import webdriver  #通过代码操控多种主流浏览器（如 Chrome、Firefox、Edge 等），自动执行用户在浏览器中能执行的任何操作
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import shutil


In [ ]:
###########  从bilibli爬取视频信息的案例
def merge_csv(input_filename, output_filename):
    """
    读取csv文件内容，并写入新的文件
    :param input_filename: 传入的文件名称
    :param output_filename: 写入的新文件的名称
    :return: 向新文件中写入input_filename中的内容
    """
 
    # 读取文件
    csv_data_read = pd.read_csv(input_filename)
    # 获取文件总行数
    number_of_row = (len(csv_data_read))
    # 循环该csv文件中的所有行，并写入信息
    for i in range(0, number_of_row):
        row_info = csv_data_read.values[i]
        # 输出查看内容
        # print(row_info)
        # 具体内容
        row_content = row_info[0]
        # 写入
        write_to_csv_bvid(output_filename, row_content)
        # 退出循环
    # 打印进度
    print(f'成功向{output_filename}中写入了{input_filename}的全部信息')
 
 
def write_to_csv_bvid(input_filename, bvid):
    """
    写入新的csv文件，若没有则创建，须根据不同程序进行修改
    :param input_filename: 写入的文件名称
    :param bvid: BV号
    :return: 生成写入的input_filename文件
    """
    # OS 判断路径是否存在
    file_exists = os.path.isfile(input_filename)
    # 设置最大尝试次数
    max_retries = 100
    retries = 0
 
    while retries < max_retries:
        try:
            with open(input_filename, mode='a', encoding='utf-8', newline='') as csvfile:
                fieldnames = ['BV号']
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
 
                if not file_exists:
                    writer.writeheader()
 
                writer.writerow({
                    'BV号': bvid
                })
                # print('写入文件成功')
            break  # 如果成功写入，跳出循环
        except PermissionError as e:
            retries += 1
            print(f"将爬取到的数据写入csv时，遇到权限错误Permission denied，文件可能被占用或无写入权限: {e}")
            print(f"等待3s后重试，将会重试50次... (尝试 {retries}/{max_retries})")
            time.sleep(3)  # 等待10秒后重试
    else:
        print("将爬取到的数据写入csv时遇到权限错误，且已达到最大重试次数50次，退出程序")
 
 
def spider_bvid(keyword):
    """
    利用seleniume获取搜索结果的bvid，供给后续程序使用
    :param keyword: 搜索关键词
    :return: 生成去重的output_filename = f'{keyword}BV号.csv'
    """
    # 保存的文件名
    input_filename = f'{keyword}BV号.csv'
 
    # 启动爬虫
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    browser = webdriver.Chrome(options=options)  # 设置无界面爬虫
    browser.set_window_size(1400, 900)  # 设置全屏，注意把窗口设置太小的话可能导致有些button无法点击
    browser.get('https://bilibili.com')
    # 刷新一下，防止搜索button被登录弹框遮住
    browser.refresh()
    print("============成功进入B站首页！！！===========")
    input = browser.find_element(By.CLASS_NAME, 'nav-search-input')
    button = browser.find_element(By.CLASS_NAME, 'nav-search-btn')
 
    # 输入关键词并点击搜索
    input.send_keys(keyword)
    button.click()
    print(f'==========成功搜索{keyword}相关内容==========')
 
    # 设置窗口
    all_h = browser.window_handles
    browser.switch_to.window(all_h[1])
    """
    # 这里可以通过xpath或者其他方法找到B站搜索结果页最下方的页码数值
    # 但B站网页代码更改后，显示为34页，网页内容检查后显示为34页（至多）
    # 由于我们的搜索结果很多，肯定超出B站最大显示的34页，故而直接设置最大页数为34
    # 找到最后一个页码所在位置，并获取值
    # total_btn = browser.find_element(By.XPATH,"//*[@id="i_cecream"]/div/div[2]/div[2]/div/div/div/div[4]/div/div/button[9]"")
    # //*[@id="i_cecream"]/div/div[2]/div[2]/div/div/div/div[4]/div/div/button[9]
    # total = int(total_btn)
    # print(f'==========成功搜索！ 总页数: {total}==========')
    """
    # B站最多显示34页
    total_page = 34
    # 同样由于B站网页代码的更改，通过找到并点击下一页的方式个人暂不能实现（对，不会分析那个破网页！！！）
    # 因此这里利用总页数进行循环访问来实现自动翻页的效果
 
    for i in range(0, total_page):
        # url 需要根据不同关键词进行调整内容！！！
        url = (f"https://search.bilibili.com/all?keyword={keyword}"
               f"&from_source=webtop_search&spm_id_from=333.1073&search_source=5&page={i}")
        print(f"===========正在尝试获取第{i + 1}页网页内容===========")
        print(f"===========本次的url为：{url}===========")
        browser.get(url)
        # 这里请求访问网页的时间也比较久，所以是否需要等待因设备而异
        # 取消刷新并长时间休眠爬虫以避免爬取太快导致爬虫抓取到js动态加载源码
        # browser.refresh()
        print('正在等待页面加载：3')
        time.sleep(2)
        print('正在等待页面加载：2')
        time.sleep(2)
        print('正在等待页面加载：1')
        time.sleep(2)
 
        # 直接分析网页
        html = browser.page_source
        # print("网页源码" + html) 用于判断是否获取成功
        soup = BeautifulSoup(html, 'lxml')
        infos = soup.find_all(class_='bili-video-card')
        bv_id_list = []
        for info in infos:
            # 只定位视频链接
            href = info.find('a').get('href')
            # 拆分
            split_url_data = href.split('/')
            # 利用循环删除拆分出现的空白
            for element in split_url_data:
                if element == '':
                    split_url_data.remove(element)
            # 打印检验内容
            # print(split_url_data)
            # 获取bvid    # Bilibili 视频的唯一标识符（Video ID），每个视频都对应一个独特的 bvid。它类似于 YouTube 的视频 ID，用来唯一标识平台上的每个视频
            bvid = split_url_data[2]
 
            # 利用if语句直接去重
            if bvid not in bv_id_list:
                bv_id_list.append(bvid)
        for bvid_index in range(0, len(bv_id_list)):
            # 写入 input_filename
            write_to_csv_bvid(input_filename, bv_id_list[bvid_index])
        # 输出提示进度
        print('写入文件成功')
        print("===========成功获取第" + str(i + 1) + "次===========")
        time.sleep(1)
        i += 1
 
    # 退出爬虫
    browser.quit()
 
    # 打印信息显示是否成功
    print(f'==========爬取完成。退出爬虫==========')
 
 
def write_to_csv(filename, bvid, aid, cid, mid, name, follower, archive, title, tname, duration,pub_date, pub_time, desc,
                 view, like, coin, favorite, share, reply, danmaku, communication_index):
    """
    向csv文件中写入B站视频相关的基本信息，未按路径找到文件，则新建文件
    :param filename: 写入数据的文件名
    :param bvid: BV号
    :param aid: AV号
    :param cid: 用于获取弹幕文本的
    :param mid: UP主的ID
    :param name: UP主名称
    :param follower: UP主粉丝数
    :param archive: UP主作品总数
    :param title: 标题
    :param tname: tag名称
    :param pub_date: 发布日期
    :param pub_time: 发布时间
    :param desc: 视频简介
    :param view: 播放量
    :param like: 点赞数
    :param coin: 投币数
    :param favorite: 收藏数
    :param share: 分享数
    :param reply: 评论数
    :param danmaku: 弹幕数
    :param communication_index: 传播效果公式的值
    :return:
    """
    file_exists = os.path.isfile(filename)
    max_retries = 100
    retries = 0
 
    while retries < max_retries:
        try:
            with open(filename, mode='a', encoding='utf-8', newline='') as csvfile:
                fieldnames = ['BV号', 'AV号', 'CID', 'UP主ID', 'UP主名称', 'UP主粉丝数', '作品总数', '视频标题',
                              '视频分类标签',"视频时长",
                              '发布日期', '发布时间', '视频简介', '播放量', '点赞数', '投币数', '收藏数', '分享数',
                              '评论数',
                              '弹幕数', '传播效果指数']
                writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
 
                if not file_exists:
                    writer.writeheader()
 
                writer.writerow({
                    'BV号': bvid, 'AV号': aid, 'CID': cid, 'UP主ID': mid, 'UP主名称': name, 'UP主粉丝数': follower,
                    '作品总数': archive, '视频标题': title, '视频分类标签': tname,'视频时长': duration, '发布日期': pub_date,
                    '发布时间': pub_time,
                    '视频简介': desc, '播放量': view, '点赞数': like, '投币数': coin, '收藏数': favorite,
                    '分享数': share,
                    '评论数': reply, '弹幕数': danmaku, '传播效果指数': communication_index
                })
            break  # 如果成功写入，跳出循环
        except PermissionError as e:
            retries += 1
            print(f"将爬取到的数据写入csv时，遇到权限错误Permission denied，文件可能被占用或无写入权限: {e}")
            print(f"等待3s后重试，将会重试50次... (尝试 {retries}/{max_retries})")
    else:
        print("将爬取到的数据写入csv时遇到权限错误，且已达到最大重试次数50次，退出程序")
 
 
def get_user_info(uid):
    """
    通过uid(即mid)获取UP主的粉丝总数和作品总数
    :param uid: mid
    :return:user_info_dict
    """
    # 定义空字典用于存放数据
    # 粉丝数 follower
    # 作品总数 archive
    user_info_dict = {}
    # 首先写入请求头
    # 设置用户代理 User_Agent及Cookies
    headers = {
        'User-Agent': "",
        'Cookie': ""}
 
    # 将传入的的uid组成up主主页的api_url
    # A Example: https://api.bilibili.com/x/web-interface/card?mid=1177893348
    api_url = f'https://api.bilibili.com/x/web-interface/card?mid={uid}'
    # https://api.bilibili.com/x/web-interface/view?BV1n24y1D75V
    # 打印次数，数据量大，便于查看进程
    print(f"正在进行爬取uid为：{uid}的UP主的粉丝数量与作品总数")
 
    # 打印本次要获取的uid，用于错误时确认
    print(f"==========本次获取数据的up主的uid为：{uid}==========")
    print(f"url为{api_url}")
 
    # 利用requests进行访问，并返回需要的封装信息
    up_info = requests.get(url=api_url, headers=headers)
 
    # 将数据转化为json格式
    up_info_json = json.loads(up_info.text)
 
    # 利用json定位相关数据
    fans_number = up_info_json['data']['card']['fans']
    user_info_dict['follower'] = fans_number
    archive_count = up_info_json['data']['archive_count']
    user_info_dict['archive'] = archive_count
 
    print(f'=========={bv_id} 的作者基本信息已成功获取==========\n')
 
    # 等待
    print('正在等待，以防访问过于频繁\n')
    time.sleep(3)
    return user_info_dict
 
def get_video_info(bv_id):
    headers = {
        'User-Agent': "",
        'Cookie': ""}
    api_url = f'https://api.bilibili.com/x/web-interface/view?bvid={bv_id}'
    # 打印本次要获取的bvid，用于错误时确认
    print(f"正在进行爬取uid为：{bv_id}的UP主的粉丝数量与作品总数")
    print(f"==========本次获取数据的视频BV号为：{bv_id}==========")
    print(f"url为：{api_url}")
    video_info = requests.get(url=api_url, headers=headers)
    video_info_json = json.loads(video_info.text)
    # 创建存放的字典
    info_dict = {}
    # 信息解读
    # https://zhuanlan.zhihu.com/p/618885790
    # 视频bvid，即bv号
    bvid = video_info_json['data']['bvid']
    info_dict['bvid'] = bvid
    # 视频aid，即av号
    aid = video_info_json['data']['aid']
    info_dict['aid'] = aid
    # 视频cid，用于获取弹幕信息
    cid = video_info_json['data']['cid'] 
    info_dict['cid'] = cid
    # 作者id
    mid = video_info_json['data']['owner']['mid']
    info_dict['mid'] = mid
    # up主昵称
    name = video_info_json['data']['owner']['name']
    info_dict['name'] = name
    # 视频标题
    title = video_info_json['data']['title']
    info_dict['title'] = title
    # 视频标签
    tname = video_info_json['data']['tname']
    info_dict['tname'] = tname
    
    # 视频时长
    duration = video_info_json['data']['duration']
    info_dict['duration'] = duration
    
    # 视频发布时间戳
    pubdate = video_info_json['data']['pubdate']
    # 转化时间戳
    pub_datatime = datetime.fromtimestamp(pubdate)
    # 整体格式
    pub_datatime_strf = pub_datatime.strftime('%Y-%m-%d %H:%M:%S')
    # 日期
    date = re.search(r"(\d{4}-\d{1,2}-\d{1,2})", pub_datatime_strf)
    info_dict['pub_date'] = date.group()
    # 时间
    pub_time = re.search(r"(\d{1,2}:\d{1,2}:\d{1,2})", pub_datatime_strf)
    info_dict['pub_time'] = pub_time.group()
    # 视频创建时间戳
    # ctime = info['ctime']
    # 视频简介
    desc = video_info_json['data']['desc']
    info_dict['desc'] = desc
    # 视频播放量
    view = video_info_json['data']['stat']['view']
    info_dict['view'] = view
    # 点赞数
    like = video_info_json['data']['stat']['like']
    info_dict['like'] = like
    # 投币数
    coin = video_info_json['data']['stat']['coin']
    info_dict['coin'] = coin
    # 收藏数
    favorite = video_info_json['data']['stat']['favorite']
    info_dict['favorite'] = favorite
    # 分享数
    share = video_info_json['data']['stat']['share']
    info_dict['share'] = share
    # 评论数
    repiy = video_info_json['data']['stat']['reply']
    info_dict['reply'] = repiy
    # 视频弹幕数量
    danmaku = video_info_json['data']['stat']['danmaku']
    info_dict['danmaku'] = danmaku
 
    print(f'=========={bv_id} 的视频基本信息已成功获取==========')
 
    # 发布作品时的动态
    # dynamic = info['dynamic']
    print('正在等待，以防访问过于频繁\n')
    time.sleep(3)
# 
    return info_dict

In [ ]:
########读取视频链接,获得BV号
import os
import csv

def merge_first_columns(directory_path, output_file):
    # 获取目录下所有文件
    files = [f for f in os.listdir(directory_path) if f.endswith('.csv')]

    # 存储所有第一列数据的列表
    merged_column_data = []

    # 遍历所有CSV文件
    for file_name in files:
        file_path = os.path.join(directory_path, file_name)

        # 打开CSV文件并读取第一列数据
        with open(file_path, 'r', newline='', encoding='utf-8') as csv_file:
            csv_reader = csv.reader(csv_file)
            # 使用next()跳过表头行
            header = next(csv_reader, None)
            first_column_data = [row[0] for row in csv_reader if row]  # 跳过空行

            # 将第一列数据添加到合并列表
            merged_column_data.extend(first_column_data)

    # 将合并的第一列数据写入新的CSV文件
    with open(output_file, 'w', newline='', encoding='utf-8') as output_csv:
        csv_writer = csv.writer(output_csv)
        for data_point in merged_column_data:
            csv_writer.writerow([data_point])

input_directory = 'test'
output_file_path = 'lianjie.csv'

# 调用函数进行合并
merge_first_columns(input_directory, output_file_path)

lianjie = pd.read_csv("lianjie.csv")
bv_id_list=[]
for i in lianjie.iloc[:,0]:
    split_url_data = i.split('/')
#     print(split_url_data)
    bvid = split_url_data[4]
#     print(bvid)
    bv_id_list.append(bvid)
        
print(len(bv_id_list))
for bvid_index in range(0, len(bv_id_list)):
# 写入 input_filename
    write_to_csv_bvid("bvnumber.csv", bv_id_list[bvid_index])
print('写入文件成功')

In [ ]:
# 遍历读取bv_id
filename = 'bvnumber.csv'
# 打开文件并去重
open_csv = pd.read_csv(filename)
open_csv
open_csv.drop_duplicates(subset='BV号')
bv_id_list = np.array(open_csv['BV号'])
print(bv_id_list)
# bv_id_list = bv_id_list[bv_id_list != 'cheese']  
# bv_id_list

In [ ]:

"""
# 第一次调用，若读取csv进行爬取时，意外中断
# 则更改为读取txt文本，将已爬取第bvid删除，以达到断点续爬的目的
for bvid in bv_id_list:
    with open("bv_id_list.txt", 'a') as f:
        f.write(bvid+'\n')
with open("bv_id_list.txt", 'r') as f:
    bv_id_list = f.readlines()
"""

# 循环写入内容
for i in range(0, len(bv_id_list)):
    bv_id = bv_id_list[i]
    print(f'正在进行第{i+1}次爬取\n')
    # 获取视频所有的基本信息
    video_info = get_video_info(bv_id)
    bvid = video_info['bvid']
    aid = video_info['aid']  ##早期用于唯一标识视频的数字 ID,在 B站 2020 年推出 bvid 之前，aid 是视频的唯一标识符
    cid = video_info['cid']  ##用来标识每个视频文件的 ID，一个视频可能有多个cid，代表不同的清晰度或不同的分集内容,接指向视频流内容。通过 cid 可以获取视频的播放数据、弹幕数据等
    mid = video_info['mid']
    name = video_info['name']
    title = video_info['title']
    tname = video_info['tname']
    duration = video_info['duration']
    pub_date = video_info['pub_date']
    pub_time = video_info['pub_time']
    desc = video_info['desc']
    view = video_info['view']
    like = video_info['like']
    coin = video_info['coin']
    favorite = video_info['favorite']
    share = video_info['share']
    reply = video_info['reply']
    danmaku = video_info['danmaku']

    # 传播效果计算公式
    Communication_Index=0
    if (0.5 * int(view) + 0.3 * (int(like) + int(coin) + int(favorite)) + 0.2 * (int(reply) + int(danmaku)))>0:
        Communication_Index = math.log(0.5 * int(view) + 0.3 * (int(like) + int(coin) + int(favorite)) + 0.2 * (int(reply) + int(danmaku)))
    # 获取作者的相关信息
    user_info = get_user_info(uid=mid)
    follower = user_info['follower']
    archive = user_info['archive']
    write_to_csv(filename='video.csv', bvid=bvid, aid=aid, cid=cid, mid=mid, name=name, follower=follower,
                 archive=archive, title=title, tname=tname,duration=duration, pub_date=pub_date, pub_time=pub_time, desc=desc,
                 view=view, like=like, coin=coin, favorite=favorite, share=share, reply=reply, danmaku=danmaku,
                 communication_index=Communication_Index)
    print(f'==========第{i+1}个BV号：{bv_id}的相关数据已写入csv文件中==========')
    print('==================================================\n')

In [ ]:
#######爬取弹幕
import csv
import re
from bs4 import BeautifulSoup
import requests
import time
'''弹幕信息参考
第一个参数是弹幕出现的时间 以秒数为单位。
第二个参数是弹幕的模式1..3 滚动弹幕 4底端弹幕 5顶端弹幕 6.逆向弹幕 7精准定位 8高级弹幕
第三个参数是字号， 12非常小,16特小,18小,25中,36大,45很大,64特别大
第四个参数是字体的颜色 以HTML颜色的十位数为准
第五个参数是Unix格式的时间戳。基准时间为 1970-1-1 08:00:00
第六个参数是弹幕池 0普通池 1字幕池 2特殊池 【目前特殊池为高级弹幕专用】
第七个参数是发送者的ID，用于“屏蔽此弹幕的发送者”功能
第八个参数是弹幕在弹幕数据库中rowID 用于“历史弹幕”功能。
目前一共9个参数，最后一个参数代表意义未知
'''


'''将弹幕时间戳转化为时间'''
def sec_to_str(second):
    second = eval(second)
    m,s = divmod(second,60)
    h,m = divmod(m,60)
    dtEventTime = "%02d:%02d:%02d" % (h,m,s)
    return dtEventTime

'''获取视频cid'''
'''采用输入bv号的方式获取弹幕信息
video_bv = input('请输入视频弹幕bv号')
url_video = 'https://www.bilibili.com/video/' + video_bv
'''

infor=pd.read_csv("video.csv")
bvnum=infor.iloc[:,0]
cids=infor.iloc[:,2]
url_videos=[]
for i in bvnum:
    url_video='https://www.bilibili.com/video/'+i
    url_videos.append(url_video)

In [ ]:
# 循环遍历每个视频的URL和对应的CID（弹幕ID）
for url, cid in zip(url_videos, cids):
    # 设置请求头，模拟浏览器访问，避免被服务器拒绝
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) '
                             'Chrome/96.0.4664.45 Safari/537.36 Edg/96.0.1054.29'}
    
    # 发送GET请求获取视频页面的内容，verify=False用来忽略SSL证书验证
    response = requests.get(url, headers=headers, verify=False)
    response.encoding = 'utf-8'  # 设置响应编码为utf-8
    html = response.content.decode()  # 将响应内容解码为字符串
    
    # 使用正则表达式从HTML中提取cid参数
    match = re.findall("cid=(.*)&aid", html)
    # print(match)  # 可选：打印匹配结果以验证
    
    '''获取弹幕文件地址'''
    # 构建弹幕文件的下载地址，使用cid拼接成弹幕文件的URL
    url_data = "https://comment.bilibili.com/" + str(cid) + ".xml"
    print(url, url_data)  # 打印视频的URL和对应的弹幕文件URL
    
    '''获取视频名称'''
    # 获取视频页面的文本内容
    page_text = response.text
    # 使用BeautifulSoup解析HTML页面
    soup = BeautifulSoup(page_text, 'html.parser')
    print(soup)  # 打印解析后的HTML内容（可选）
    
    # 查找视频标题，使用BeautifulSoup定位特定的HTML元素
    # video_title_low = soup.find('span', class_='tit').text
    # 处理标题中的标点符号（目前代码中未实际应用）
    punctuation = r"""|"""
    dicts = {i: '' for i in punctuation}
    punc_table = str.maketrans(dicts)
    # video_title = video_title_low.translate(punc_table)  # 原代码尝试删除标点符号
    video_title = str(cid)  # 使用cid作为视频标题（方便后续操作）
    print(video_title)  # 打印视频标题
    
    '''获取弹幕文件'''
    # 发送GET请求获取弹幕文件的内容
    res = requests.get(url_data, headers=headers, verify=False)
    res.encoding = 'utf-8'  # 设置响应编码为utf-8
    page_text = res.text  # 获取弹幕文件的文本内容
    # 使用BeautifulSoup解析弹幕文件的XML内容
    soup = BeautifulSoup(page_text, 'html.parser')
    # barrages = [re.sub(r'\s+', '', bar.text) for bar in soup.find_all('d')]  # 可选：去除弹幕中的多余空白字符
    barrages = soup.find_all('d')  ### 查找所有弹幕标签（<d>）
    
    # 初始化一个列表来存储所有弹幕信息
    all_list = []
    for item in barrages:
        # 获取弹幕的属性信息，并将其分割成列表
        barrage_list = item.get('p').split(",")
        # 将弹幕文本信息添加到列表中
        barrage_list.append(item.string)
        # 将弹幕的时间戳转换为可读的时间格式
        barrage_list[4] = time.ctime(eval(barrage_list[4]))
        # 将处理好的弹幕信息添加到总列表中
        all_list.append(barrage_list)
    
    '''将弹幕信息写入文件中'''
    # 构建保存文件的名称，使用视频标题（cid）加.csv后缀
    file_name = video_title + '.csv'
    
    # 定义CSV文件的表头
    tableheader = ['弹幕出现时间', '弹幕格式', '弹幕字体', '弹幕颜色', '弹幕时间戳',
                   '弹幕池', '用户ID', 'rowID', '未知信息', '弹幕信息']
    
    '''这个是只爬取弹幕内容
    with open(file_name, 'w', encoding='utf-8') as output_file:
        for bar in barrages:
            output_file.write(bar + '\n')
    '''
    
    '''这个是爬取弹幕所有信息'''
    # 以追加模式打开CSV文件，如果文件不存在则创建，并忽略编码错误
    with open(file_name, 'a', newline='', errors='ignore', encoding='utf-8') as fd:
        writer = csv.writer(fd)  # 创建CSV写入器
        writer.writerow(tableheader)  # 写入表头
        for row in all_list:
            # 为防止rowID列用科学计数法表示，添加一个制表符'\t'
            row[7] = str(row[7]) + '\t'
            writer.writerow(row)  # 写入每一行弹幕信息
    
    print('弹幕信息已成功写入文件！')  # 提示写入成功

In [18]:
# 指定源文件夹路径（包含CSV文件的文件夹）
source_folder = 'D:/桌面/最终作业'
# 指定目标文件夹路径（要将文件移动到的文件夹）
target_folder = 'D:/桌面/最终作业/b站弹幕集合'

# 检查目标文件夹是否存在，如果不存在则创建
if not os.path.exists(target_folder):
    os.makedirs(target_folder)

# 遍历源文件夹中的所有文件
for file_name in os.listdir(source_folder):
    # 检查文件名是否以数字开头且以.csv结尾
    if file_name[0].isdigit() and file_name.endswith('.csv'):
        # 构建源文件的完整路径
        source_file_path = os.path.join(source_folder, file_name)
        # 构建目标文件的完整路径
        target_file_path = os.path.join(target_folder, file_name)
        
        # 打印文件名以确认匹配
        print(f"准备移动文件：{file_name} 从 {source_folder} 到 {target_folder}")

        # 尝试移动文件
        try:
            shutil.move(source_file_path, target_file_path)
            print(f"文件 {file_name} 已成功移动到 {target_folder}")
        except Exception as e:
            print(f"文件 {file_name} 移动失败: {e}")
    else:
        print(f"文件 {file_name} 不符合移动条件")

print("文件移动操作完成。")

文件 .idea 不符合移动条件
文件 bvnumber.csv 不符合移动条件
文件 b站弹幕集合 不符合移动条件
文件 Crawler-blili.ipynb 不符合移动条件
文件 lianjie.csv 不符合移动条件
文件 test 不符合移动条件
文件 video.csv 不符合移动条件
文件 数据爬取.ipynb 不符合移动条件
文件移动操作完成。


In [19]:
# 指定包含CSV文件的文件夹路径
folder_path = r'D:/桌面/最终作业/b站弹幕集合'  # 使用原始字符串

try:
    all_files = os.listdir(folder_path)
except Exception as e:
    print(f"无法读取文件夹：{e}")
    exit()

# 读取所有CSV文件并存储在一个列表中
csv_files = [os.path.join(folder_path, f) for f in all_files if f.endswith('.csv')]

# 创建一个空的DataFrame用于存储合并后的数据
merged_csv = pd.DataFrame()

try:
    # 遍历列表中的每个CSV文件路径
    for file in csv_files:
        df = pd.read_csv(file)
        # 将读取的数据追加到merged_csv DataFrame中
        merged_csv = pd.concat([merged_csv, df], ignore_index=True)
except Exception as e:
    print(f"读取CSV文件时出错：{e}")
    exit()

# 将合并后的数据保存为新的CSV文件
output_path = 'merged_output.csv'  # 你希望保存合并后CSV的文件名
try:
    merged_csv.to_csv(output_path, index=False)
    print(f"合并后的CSV文件已保存到：{output_path}")
except Exception as e:
    print(f"保存CSV文件时出错：{e}")

合并后的CSV文件已保存到：merged_output.csv
